In [1]:
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk
import random


class NimGameApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Nim Game - Human vs AI (OOP)")
        self.root.geometry("720x420")

        # -------- State --------
        self.piles = [3, 4, 5]
        self.undo_stack = []
        self.history_lines = []
        self.human_wins = 0
        self.ai_wins = 0

        self.ai_thinking = tk.BooleanVar(value=False)
        self.selected_pile = tk.IntVar(value=0)
        self.difficulty = tk.StringVar(value="Hard")
        self.first_player = tk.StringVar(value="Human")
        self.status_kind = tk.StringVar(value="info")

        self.pile_labels = [tk.StringVar(), tk.StringVar(), tk.StringVar()]
        self.status = tk.StringVar()
        self.score_var = tk.StringVar()

        # -------- UI --------
        self._build_ui()
        self._bind_keys()

        # Start
        self.set_status("AI plays based on difficulty. Try your luck 😈", "info")
        self.highlight_selected_pile(0)
        self.start_new_game([3, 4, 5])

    # =======================
    #        GAME LOGIC
    # =======================
    def nim_sum(self, piles):
        return piles[0] ^ piles[1] ^ piles[2]

    def optimal_ai_move(self, piles_local):
        x = self.nim_sum(piles_local)
        if x == 0:
            return None
        for i in range(3):
            if (piles_local[i] ^ x) < piles_local[i]:
                new_val = piles_local[i] ^ x
                removed = piles_local[i] - new_val
                return (i, removed)
        return None

    def random_ai_move(self, piles_local):
        non_empty = [i for i, p in enumerate(piles_local) if p > 0]
        if not non_empty:
            return None
        i = random.choice(non_empty)
        removed = random.randint(1, piles_local[i])
        return (i, removed)

    def medium_ai_move(self, piles_local):
        if random.random() < 0.6:
            m = self.optimal_ai_move(piles_local)
            if m is not None:
                return m
        return self.random_ai_move(piles_local)

    def ai_choose_move(self, piles_local):
        d = self.difficulty.get()
        if d == "Easy":
            return self.random_ai_move(piles_local)
        if d == "Medium":
            return self.medium_ai_move(piles_local)

        m = self.optimal_ai_move(piles_local)
        return m if m is not None else self.random_ai_move(piles_local)

    # =======================
    #       TURN FLOW
    # =======================
    def human_move(self, pile_index: int):
        if self.ai_thinking.get():
            return

        self.selected_pile.set(pile_index)
        self.highlight_selected_pile(pile_index)

        try:
            stones = int(self.entry.get())
        except:
            self.set_status("Enter a valid number!", "error")
            return

        if stones < 1 or stones > self.piles[pile_index]:
            self.set_status("❌ Invalid move!", "error")
            return

        # undo snapshot BEFORE human move
        self.push_undo_state()

        self.piles[pile_index] -= stones
        self.add_history(f"Human removed {stones} from pile {chr(65 + pile_index)}")
        self.set_status(f"You removed {stones} stones from pile {chr(65 + pile_index)}", "info")
        self.update_ui()

        if self.check_game_over(after_ai=False):
            return

        self.start_ai_turn()

    def start_ai_turn(self):
        self.ai_thinking.set(True)
        self.disable_buttons()

        self.progress.pack(pady=6)
        self.progress.start(12)
        self.set_status("AI thinking...", "ai")

        self.root.after(450, self.apply_ai_move)

    def apply_ai_move(self):
        self.stop_ai_thinking()

        move = self.ai_choose_move(self.piles[:])
        if move is None:
            self.check_game_over(after_ai=True)
            return

        i, removed = move
        self.piles[i] -= removed

        self.add_history(f"AI removed {removed} from pile {chr(65 + i)}")
        self.set_status(f"AI removed {removed} stones from pile {chr(65 + i)}", "ai")
        self.update_ui()

        if self.check_game_over(after_ai=True):
            return

        self.enable_buttons()

    def check_game_over(self, after_ai: bool):
        if not all(p == 0 for p in self.piles):
            return False

        if after_ai:
            self.ai_wins += 1
            self.update_scoreboard()
            messagebox.showinfo("Game Over", "💀 AI wins!")
        else:
            self.human_wins += 1
            self.update_scoreboard()
            messagebox.showinfo("Game Over", "🎉 You win!")

        self.end_round()
        return True

    def end_round(self):
        self.disable_buttons()
        self.progress.stop()
        self.progress.pack_forget()

    # =======================
    #       UNDO + LOG
    # =======================
    def snapshot_state(self):
        return {
            "piles": self.piles[:],
            "selected": self.selected_pile.get(),
            "status": self.status.get(),
            "status_kind": self.status_kind.get(),
            "history_len": len(self.history_lines),
        }

    def push_undo_state(self):
        self.undo_stack.append(self.snapshot_state())

    def undo_move(self):
        if not self.undo_stack:
            self.set_status("Nothing to undo!", "error")
            return

        st = self.undo_stack.pop()
        self.piles = st["piles"][:]
        self.selected_pile.set(st["selected"])
        self.highlight_selected_pile(st["selected"])
        self.set_status(st["status"], st["status_kind"])

        # Trim history to previous length
        self.history_lines = self.history_lines[: st["history_len"]]
        self.rebuild_history()

        self.update_ui()

    def add_history(self, line: str):
        self.history_lines.append(line)
        self.history_text.config(state="normal")
        self.history_text.insert("end", line + "\n")
        self.history_text.config(state="disabled")
        self.history_text.see("end")

    def rebuild_history(self):
        self.history_text.config(state="normal")
        self.history_text.delete("1.0", "end")
        for line in self.history_lines:
            self.history_text.insert("end", line + "\n")
        self.history_text.config(state="disabled")
        self.history_text.see("end")

    # =======================
    #       UI HELPERS
    # =======================
    def update_ui(self):
        self.pile_labels[0].set(f"A: {self.piles[0]}")
        self.pile_labels[1].set(f"B: {self.piles[1]}")
        self.pile_labels[2].set(f"C: {self.piles[2]}")
        self.entry.delete(0, tk.END)
        self.update_buttons_state()

    def set_status(self, msg: str, kind: str = "info"):
        self.status.set(msg)
        self.status_kind.set(kind)
        if kind == "error":
            self.status_label.config(fg="red")
        elif kind == "win":
            self.status_label.config(fg="green")
        elif kind == "ai":
            self.status_label.config(fg="purple")
        else:
            self.status_label.config(fg="blue")

    def disable_buttons(self):
        for b in self.pile_buttons:
            b.config(state="disabled")
        self.undo_btn.config(state="disabled")
        self.enter_btn.config(state="disabled")

    def enable_buttons(self):
        for b in self.pile_buttons:
            b.config(state="normal")
        self.undo_btn.config(state="normal")
        self.enter_btn.config(state="normal")
        self.update_buttons_state()

    def update_buttons_state(self):
        for i, b in enumerate(self.pile_buttons):
            if self.piles[i] == 0 or self.ai_thinking.get():
                b.config(state="disabled")
            else:
                b.config(state="normal")

    def highlight_selected_pile(self, i: int):
        for idx, b in enumerate(self.pile_buttons):
            b.config(relief="sunken" if idx == i else "raised")

    def stop_ai_thinking(self):
        self.ai_thinking.set(False)
        self.progress.stop()
        self.progress.pack_forget()

    # =======================
    #   EXTRA GUI FUNCTIONS
    # =======================
    def reset_game(self):
        self.start_new_game([3, 4, 5])

    def random_piles(self):
        self.start_new_game([random.randint(1, 15), random.randint(1, 15), random.randint(1, 15)])

    def set_custom_piles(self):
        try:
            a = int(self.pileA_entry.get())
            b = int(self.pileB_entry.get())
            c = int(self.pileC_entry.get())
        except:
            self.set_status("Custom piles must be numbers!", "error")
            return
        if a < 1 or b < 1 or c < 1:
            self.set_status("Custom piles must be >= 1", "error")
            return
        self.start_new_game([a, b, c])

    def apply_first_player(self):
        # Restart with current piles, applying first player
        self.start_new_game(self.piles[:])

    def show_rules(self):
        messagebox.showinfo(
            "Rules",
            "Nim Rules:\n"
            "• 3 piles (A, B, C)\n"
            "• On your turn, choose ONE pile and remove 1+ stones\n"
            "• Whoever takes the LAST stone wins\n\n"
            "Controls:\n"
            "• Press A/B/C to select pile\n"
            "• Type number and press Enter to move\n"
        )

    def on_difficulty_change(self, _=None):
        self.set_status(f"Difficulty set to: {self.difficulty.get()}", "info")

    def update_scoreboard(self):
        self.score_var.set(f"Score — Human: {self.human_wins} | AI: {self.ai_wins}")

    def start_new_game(self, new_piles):
        self.stop_ai_thinking()

        self.piles = new_piles[:]
        self.undo_stack.clear()
        self.history_lines = []

        self.rebuild_history()
        self.add_history(f"--- New Game: A={self.piles[0]}, B={self.piles[1]}, C={self.piles[2]} ---")

        self.selected_pile.set(0)
        self.highlight_selected_pile(0)
        self.set_status("Game started. Good luck 😈", "info")

        self.update_ui()
        self.enable_buttons()

        if self.first_player.get() == "AI":
            self.start_ai_turn()

    # =======================
    #        KEY BINDS
    # =======================
    def _bind_keys(self):
        self.root.bind("a", lambda e: self.select_pile(0))
        self.root.bind("b", lambda e: self.select_pile(1))
        self.root.bind("c", lambda e: self.select_pile(2))
        self.root.bind("<Return>", lambda e: self.human_move(self.selected_pile.get()))

    def select_pile(self, i: int):
        self.selected_pile.set(i)
        self.highlight_selected_pile(i)
        self.entry.focus_set()

    # =======================
    #          UI BUILD
    # =======================
    def _build_ui(self):
        tk.Label(self.root, text="🎮 Nim Game (Human vs AI)", font=("Arial", 14, "bold")).pack(pady=6)

        top = tk.Frame(self.root)
        top.pack(fill="x", padx=10)

        tk.Label(top, text="Difficulty:").grid(row=0, column=0, sticky="w")
        self.diff_menu = ttk.Combobox(top, textvariable=self.difficulty,
                                      values=["Easy", "Medium", "Hard"], width=10, state="readonly")
        self.diff_menu.grid(row=0, column=1, padx=6)
        self.diff_menu.bind("<<ComboboxSelected>>", self.on_difficulty_change)

        tk.Label(top, text="First Player:").grid(row=0, column=2, sticky="w", padx=(12, 0))
        self.fp_menu = ttk.Combobox(top, textvariable=self.first_player,
                                    values=["Human", "AI"], width=10, state="readonly")
        self.fp_menu.grid(row=0, column=3, padx=6)

        tk.Button(top, text="Apply First Player", command=self.apply_first_player).grid(row=0, column=4, padx=6)
        tk.Button(top, text="Rules", command=self.show_rules).grid(row=0, column=5, padx=6)

        self.update_scoreboard()
        tk.Label(self.root, textvariable=self.score_var, font=("Arial", 11, "bold")).pack(pady=4)

        piles_frame = tk.Frame(self.root)
        piles_frame.pack(pady=6)

        tk.Label(piles_frame, textvariable=self.pile_labels[0], font=("Arial", 12)).grid(row=0, column=0, padx=10)
        tk.Label(piles_frame, textvariable=self.pile_labels[1], font=("Arial", 12)).grid(row=0, column=1, padx=10)
        tk.Label(piles_frame, textvariable=self.pile_labels[2], font=("Arial", 12)).grid(row=0, column=2, padx=10)

        tk.Label(self.root, text="Stones to remove:", font=("Arial", 10)).pack(pady=4)
        self.entry = tk.Entry(self.root, width=10)
        self.entry.pack()

        btn_frame = tk.Frame(self.root)
        btn_frame.pack(pady=8)

        self.pile_buttons = []
        self.pile_buttons.append(tk.Button(btn_frame, text="Remove from A", width=14, command=lambda: self.human_move(0)))
        self.pile_buttons.append(tk.Button(btn_frame, text="Remove from B", width=14, command=lambda: self.human_move(1)))
        self.pile_buttons.append(tk.Button(btn_frame, text="Remove from C", width=14, command=lambda: self.human_move(2)))

        self.pile_buttons[0].grid(row=0, column=0, padx=5)
        self.pile_buttons[1].grid(row=0, column=1, padx=5)
        self.pile_buttons[2].grid(row=0, column=2, padx=5)

        self.enter_btn = tk.Button(btn_frame, text="Enter (Selected Pile)", width=18,
                                   command=lambda: self.human_move(self.selected_pile.get()))
        self.enter_btn.grid(row=0, column=3, padx=8)

        self.undo_btn = tk.Button(btn_frame, text="Undo", width=10, command=self.undo_move)
        self.undo_btn.grid(row=0, column=4, padx=5)

        self.progress = ttk.Progressbar(self.root, mode="indeterminate", length=220)

        self.status_label = tk.Label(self.root, textvariable=self.status, fg="blue")
        self.status_label.pack(pady=6)

        bottom = tk.Frame(self.root)
        bottom.pack(fill="x", padx=10, pady=6)

        tk.Button(bottom, text="Reset (3,4,5)", command=self.reset_game).grid(row=0, column=0, padx=5)
        tk.Button(bottom, text="New Random", command=self.random_piles).grid(row=0, column=1, padx=5)

        tk.Label(bottom, text="Custom A,B,C:").grid(row=0, column=2, padx=(18, 4))
        self.pileA_entry = tk.Entry(bottom, width=5)
        self.pileB_entry = tk.Entry(bottom, width=5)
        self.pileC_entry = tk.Entry(bottom, width=5)
        self.pileA_entry.grid(row=0, column=3, padx=2)
        self.pileB_entry.grid(row=0, column=4, padx=2)
        self.pileC_entry.grid(row=0, column=5, padx=2)

        tk.Button(bottom, text="Start Custom", command=self.set_custom_piles).grid(row=0, column=6, padx=6)

        log_frame = tk.Frame(self.root)
        log_frame.pack(fill="both", expand=True, padx=10, pady=6)

        tk.Label(log_frame, text="Move History:").pack(anchor="w")
        self.history_text = tk.Text(log_frame, height=7, wrap="word", state="disabled")
        self.history_text.pack(fill="both", expand=True)


if __name__ == "__main__":
    root = tk.Tk()
    app = NimGameApp(root)
    root.mainloop()
